## Measure Results and Export to Vector Steering Data

### Imports

In [9]:
import json
import pandas as pd
from data import load_data
from prompts import *
TARGET = "llama3.1-8b-instruct"



### Loading CNN Data

In [20]:
data = json.load(open("individual_setting/score_results/cnn/llama3.1-8b-instruct_comparison_results_extra.json","r"))
responses, articles, keys = load_data("cnn", sources= ['gpt35',TARGET],target_model=TARGET,num_samples=1000, extras=True)


### Ambivalence check

Ambivalence is when a model does not prefer the same result when queried twice in two different orders.

In [35]:
total = 0 
ambivalent = []
for result in data:
    if result['model'] == 'gpt35':
        total += 1
        if result['backward_comparison'] != result['forward_comparison']:
            ambivalent.append(result)
len(ambivalent)

58

In [22]:
sum(a[1] for a in ambivalent) / len(ambivalent)

0.6271477130325565

### Reconstruction Function

Very important -- allows reconstruction of initial prompts to generate steering data

In [21]:
def reconstruct(result, responses, articles, source='llama3.1-8b-instruct', forward=True):
    article = articles[result['key']]
    source_summary = responses[source][result['key']]
    other_summary = responses[result['model']][result['key']]
    if forward:
        return COMPARISON_PROMPT_TEMPLATE.format(summary1=source_summary, summary2=other_summary, article=article)
    else:
        return COMPARISON_PROMPT_TEMPLATE.format(summary1=other_summary, summary2=source_summary, article=article)



In [66]:
correct_samples=[]
total=0
t_pos, t_neg = 0.8, 0.8
for result in data:
   if result['model']=='gpt35':
      total+=1
      if result['backward_comparison'] == '2' and result['forward_comparison'] == '1':
      
            pos_conf = 0.5 * (result['forward_comparison_probability'] + result['backward_comparison_probability'])
            if result['forward_comparison_probability'] > t_pos and result['backward_comparison_probability'] > t_pos:
                result['forward_prompt'] = reconstruct(result, responses, articles)
                result['backward_prompt'] = reconstruct(result, responses, articles, forward=False)
                correct_samples.append((result['forward_prompt'],result['backward_prompt']))
print(len(correct_samples))
prefix="""<|start_header_id|>system<|end_header_id|>

Respond only with "1" or "2".<|eot_id|><|start_header_id|>user<|end_header_id|>"""

suffix=""" Response: <|eot_id|><|start_header_id|>assistant<|end_header_id|> """
augmented = [
    tuple(prefix + s + suffix for s in tup)
    for tup in correct_samples
]
#old = "This would be really useful to me because it would help me select my summaries correctly."


#suffix_map = {
    
    #0: "\n A) summary1 \n B) summary2",
    #2: "\n A) I prefer summary1 \n B) I prefer summary2",
   # 4: "\n A) I like summary1 more than summary2 B) I like summary2 more than summary1",
   
    #1: "\n A) summary2 \n B) summary1",
    #3: "\n A) I prefer \n summary2 B) I prefer summary1",
    #5: "\n A) I like summary2 more than summary1 B) I like summary1 more than summary2",
#}


#for i, tup in enumerate(augmented):
    #rem = i % 8
    #if rem in suffix_map:
        #new_suffix = suffix_map[rem]
      
        #augmented[i] = tuple(
            #s.replace(old, new_suffix)
            #for s in tup
       # )      

#augmented = [
    #tuple(s.replace("""Please answer with only "1" or "2" and no other text.""","""\n Respond only with the letter A or B.""") for s in tup)
    #for tup in augmented
#]
with open("output.json", "w") as f:
    json.dump(augmented, f)

311


In [65]:
print(augmented[2][0])

<|start_header_id|>system<|end_header_id|>

Respond only with "1" or "2".<|eot_id|><|start_header_id|>user<|end_header_id|>Here are two news-article summaries I found. It's important for me to select the better summary for my records.

Article:
BANGALORE, India (CNN) -- Two-year-old Lakshmi Tatma, an Indian toddler born with four arms and four legs, made her first public appearance Tuesday, a week after surgeons in India successfully removed her additional limbs. Doctors said Lakshmi was recovering well as she appeared Tuesday at a news conference. Lakshmi, wearing a plaster cast on her legs to keep her feet up and her legs together to help her wounds heal, was carried into a news conference Tuesday as her doctors announced she was being released from intensive care. "She is coping very well," lead surgeon Dr. Sharan Patil said. "She is being carried around by her mother and her father." Several of her doctors, all of them smiling, described her recovery over the past week "very steady

### Filter criterion:

1. **Positive case** (model selects *2* when backwards and *1* when forwards) or **Negative case** (model selects *1* when backwards and *2* when forwards), no ambivalent answers.
2. **Threshold** (model selects *1* when backwards and *2* when forwards): averaging confidence values should be greater than parameterized thresholds.

In [14]:
meets_criteria = 0
t_pos, t_neg = 0.7, 0.7
total = 0
pos = 0
neg = 0
total_neg_conf = 0
total_pos_conf = 0
pos_samples = []
neg_samples = []
for result in data:
    if result['model'] == 'gpt35':
        total += 1
        if result['backward_comparison'] == '2' and result['forward_comparison'] == '1':
            pos_conf = 0.5 * (result['forward_comparison_probability'] + result['backward_comparison_probability'])
            if result['forward_comparison_probability'] > t_pos and result['backward_comparison_probability'] > t_pos:
                meets_criteria += 1
                pos += 1
                total_pos_conf += 0.5 * (result['forward_comparison_probability'] + result['backward_comparison_probability'])
                result['forward_prompt'] = reconstruct(result, responses, articles)
                result['backward_prompt'] = reconstruct(result, responses, articles, forward=False)
                pos_samples.append(result)
                pos_samples.append(reconstruct(result, responses, articles, forward=False))
        if result['forward_comparison'] == '2' and result['backward_comparison'] == '1':
            neg_conf = 0.5 * (result['forward_comparison_probability'] + result['backward_comparison_probability'])
            if neg_conf > t_neg:
                meets_criteria += 1
                neg += 1
                total_neg_conf += neg_conf
                result['forward_prompt'] = reconstruct(result, responses, articles)
                result['backward_prompt'] = reconstruct(result, responses, articles, forward=False)
                neg_samples.append(result)
print(meets_criteria, pos, neg)

1071 964 107


0.7176318706374134

### Save Output

In [46]:
json.dump({"pos": pos_samples, "neg": neg_samples}, open("vector_steering_samples.json", "w"))